In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("--- APPROACH 1: BASELINE RANDOM FOREST ---")
# Data loading & prep (same as File 1)
df = pd.read_csv('combined_stock_data.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['High'] = df.groupby('Company Name')['High'].ffill()
df['Low'] = df.groupby('Company Name')['Low'].ffill()
df['Volume'] = df['Volume'].fillna(0)
df['Day_Range'] = df['High'] - df['Low']
df['Daily_Return'] = df.groupby('Company Name')['Close'].pct_change() * 100
df['Target_Trend'] = (df['Close'] > df['Open']).astype(int)

features = ['Open', 'High', 'Low', 'Volume', 'Day_Range', 'Daily_Return']
df_clean = df.dropna(subset=features + ['Target_Trend']).copy()
X = df_clean[features]
y = df_clean['Target_Trend']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

rf_baseline = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_baseline.fit(X_train, y_train)

baseline_acc = accuracy_score(y_test, rf_baseline.predict(X_test))
print(f"Baseline Accuracy (All Features): {baseline_acc * 100:.2f}%")

--- APPROACH 1: BASELINE RANDOM FOREST ---
Baseline Accuracy (All Features): 84.69%


In [2]:
print("\n--- APPROACH 2: FEATURE ABLATION STUDY ---")
print("We will loop through and remove one column at a time to see how much the accuracy drops. This proves which features are most important.")

for col in features:
    # Drop one column for this test
    X_train_dropped = X_train.drop(columns=[col])
    X_test_dropped = X_test.drop(columns=[col])
    
    # Train new model
    rf_ablation = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    rf_ablation.fit(X_train_dropped, y_train)
    
    # Grade new model
    new_accuracy = accuracy_score(y_test, rf_ablation.predict(X_test_dropped))
    
    # Calculate impact
    impact = baseline_acc - new_accuracy
    print(f"Removing '{col:<12}' -> Accuracy: {new_accuracy * 100:.2f}% (Impact: {impact * 100:+.2f}%)")


--- APPROACH 2: FEATURE ABLATION STUDY ---
We will loop through and remove one column at a time to see how much the accuracy drops. This proves which features are most important.
Removing 'Open        ' -> Accuracy: 84.66% (Impact: +0.03%)
Removing 'High        ' -> Accuracy: 84.67% (Impact: +0.02%)
Removing 'Low         ' -> Accuracy: 84.67% (Impact: +0.03%)
Removing 'Volume      ' -> Accuracy: 84.74% (Impact: -0.05%)
Removing 'Day_Range   ' -> Accuracy: 84.64% (Impact: +0.05%)
Removing 'Daily_Return' -> Accuracy: 57.48% (Impact: +27.21%)
